# AstroMed V3 · Dalillarni tahlil qilish (DEMO)

Kaggle’da GPU va Internet’ni yoqing, so‘ng Run All. Notebook faqat bitta GPU’ni ko‘radi.
Har bir ehtimol alohida baholanadi. Dalil IDlari va javob shakli kodda tekshiriladi.
Simulyator ogohlantirishi model javobidan mustaqil ishlaydi.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "peft==0.17.1", "accelerate==1.10.1",
    "bitsandbytes==0.48.1",
])

In [ ]:
import json
import random
from itertools import permutations

TOPICS = ("muhit", "ovqat", "sensor", "parvoz")
ACTIONS = dict(zip(TOPICS, ("CHECK_CABIN", "CHECK_INTAKE", "VERIFY_SENSOR", "CHECK_FLIGHT")))
ACTIONS.update(yetarli_emas="ASK_CONTEXT", ziddiyat="REVIEW_CONFLICT")
SYSTEM = (
    "AstroMed DEMO. Har bir ehtimolni tekshir va faqat JSON ber. "
    "ehtimollar obyektida muhit, ovqat, sensor, parvoz bo‘lsin. "
    "Har birida holat qiymati mos, qarshi yoki noma_lum va dalil degan ro‘yxat bo‘lsin. "
    "mos yoki qarshi bo‘lsa faqat tegishli qayd IDlarini ber; noma_lum bo‘lsa bo‘sh ro‘yxat. "
    "Qayd yo‘qligi va qarshi dalil boshqa-boshqa holatlar. Tashxis yoki muolaja yozma."
)

# Each pair varies the evidence for exactly one hypothesis.
TRAIN = {
    "muhit": [
        ("Shikoyatlar ayni modulda boshlangan; ventilyatsiya o‘sha payt o‘chgan.",
         "Shikoyatlar paytida ventilyatsiya ishlagan va sifatli kabina trendlari barqaror."),
        ("Bir xonadagi ekipaj shikoyat qilgan paytda bosim sezilarli o‘zgargan.",
         "Mustaqil bosim datchigi esa shu davrda barqaror bosimni tasdiqlagan."),
        ("Bir zonada alomat boshlanganida sifatli CO₂ qaydi o‘zgargan.",
         "Alomatlar turli zonalarda boshlangan va CO₂ qaydlari barqaror."),
    ],
    "ovqat": [
        ("Shikoyat qilganlar ayni partiyadagi ichimlikdan ichgan, boshqalar ichmagan.",
         "Shikoyat qilganlar turli manbadan ichgan, bir xil ichimlik ichishmagan."),
        ("Faqat bir xil taomni yeganlar undan keyin shikoyat bildirgan.",
         "Shikoyatlar qayd etilgan taomni iste’mol qilishdan oldin boshlangan."),
        ("Turli kabinalardagi shikoyat qilganlar ayni suv idishidan foydalangan.",
         "Shikoyat qilganlar umumiy suv yoki ovqat iste’mol qilmagan."),
    ],
    "sensor": [
        ("Shubhali o‘lchov eski; ayni paytdagi mustaqil sensor o‘zgarishni takrorlamagan.",
         "Ikkita mustaqil va joriy sifatli sensor bir xil o‘zgarishni tasdiqlagan."),
        ("O‘lchovda uzilish va sakrash bor; tekshiruvda batareya nosoz chiqqan.",
         "Mustaqil o‘lchagich signalni tasdiqlagan, qurilma sinovi ham o‘tgan."),
        ("Faqat sifat bayrog‘i past bo‘lgan sensorda sakrash bor.",
         "Yangi sifatli muqobil sensor ham shu sakrashni ko‘rsatgan."),
    ],
    "parvoz": [
        ("Qo‘ngandan keyingi ilk tik turishda alomat qayta-qayta paydo bo‘lgan.",
         "Alomat tik turishdan oldin boshlangan va holat almashganda o‘zgarmagan."),
        ("Orbitaning ilk kunlarida alomat faqat bosh tez burilganda takrorlangan.",
         "Alomat bosh harakati bilan takrorlanmagan va orbitaga chiqishdan oldin boshlangan."),
        ("Og‘ir skafandr ishining yakunida alomat ayni yuklamada yana qayd etilgan.",
         "Alomat skafandrga kirishdan oldin qayd etilgan va yuklama bilan o‘zgarmagan."),
    ],
}
HELD = {
    "muhit": ("LAB’dagi ikki a’zo alomat yozgan soatda shu modulda havo aylanishi uzilgan.",
               "LAB’dagi o‘sha soatda havo aylanishi va mustaqil kabina o‘lchovlari barqaror."),
    "ovqat": ("Ikki moduldagi a’zolar ayni qadoqdagi sharbatni ichgach bir paytda shikoyat qilgan.",
               "Qadoq ochilmasidan avval ikkala a’zoning shikoyati jurnalga tushgan."),
    "sensor": ("Puls sakrashi yozilgan kanal sinxronlashmagan; ko‘chma o‘lchagichda sakrash yo‘q.",
                "Ko‘chma o‘lchagich ham sinxron va sifatli puls yozuvida ayni sakrashni tasdiqlagan."),
    "parvoz": ("Yerga qaytgan a’zoda shikoyat yotgan joyidan o‘tirishga o‘tganda takrorlangan.",
                "Yerga qaytgan a’zoda holat o‘zgarishidan qat’i nazar alomat bir xil bo‘lgan."),
}
MISSING = {
    "muhit": "Kabina telemetriyasi mavjud emas.",
    "ovqat": "Iste’mol jurnali to‘ldirilmagan.",
    "sensor": "Sensor sifati va muqobil o‘lchov haqida qayd yo‘q.",
    "parvoz": "Parvoz bosqichi va faoliyat vaqti yozilmagan.",
}

def make_case(clues, number, symptom, family, alert=False):
    rng = random.Random(number)
    layout = rng.sample(range(1, 20), 5)
    records = {f"E{layout[0]}": symptom}
    expected = {}
    for i, topic in enumerate(TOPICS, start=1):
        status, statement = clues[topic]
        ref = f"E{layout[i]}"
        records[ref] = statement
        expected[topic] = {"holat": status, "dalil": [ref] if status != "noma_lum" else []}
    items = list(records.items())
    rng.shuffle(items)
    return {
        "family": family,
        "records": records,
        "alert": alert,
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": "\n".join(f"{key}: {value}" for key, value in items)},
        ],
        "expected": {"ehtimollar": expected},
    }

def clues_for(pairs, supported, opposing):
    return {key: ("mos", pairs[key][0]) if key in supported else
            ("qarshi", pairs[key][1]) if key in opposing else
            ("noma_lum", MISSING[key]) for key in TOPICS}

def make_split(pairs, seed, symptom):
    rows = []
    for first, second in permutations(TOPICS, 2):
        # Same symptom, IDs and record order in all four conditions.
        family = f"{seed}:{first}:{second}"
        for supported in ((), (first,), (second,), (first, second)):
            clues = clues_for(pairs, supported, {first, second} - set(supported))
            rows.append(make_case(clues, seed + len(rows)//4, symptom, family))
    rows.append(make_case(clues_for(pairs, (), ()), seed+50, symptom, f"{seed}:missing"))
    return rows

train = []
for variant, symptom in enumerate([
    "Ikki a’zo bosh aylanishini qayd etdi.",
    "Ekipaj jurnalida ko‘ngil aynishi qayd etilgan.",
    "Smena davomida holsizlik haqida xabar berilgan.",
]):
    train.extend(make_split({key: TRAIN[key][variant] for key in TOPICS}, 1000+variant*100, symptom))
random.Random(42).shuffle(train)
test = make_split(HELD, 3000, "Ekipaj bosh aylanishi va holsizlik haqida xabar bergan.")
assert {r["family"] for r in train}.isdisjoint(r["family"] for r in test)
print(f"O‘quv: {len(train)}; yangi sinov: {len(test)}.")

def choose_action(parsed, records, alert):
    # This alert flag belongs to the simulator, outside the model input/output.
    if alert:
        return "SHOW_EXISTING_ALERT"
    if not isinstance(parsed, dict) or set(parsed) != {"ehtimollar"}:
        return "REVIEW_MODEL_OUTPUT"
    hypotheses = parsed["ehtimollar"]
    if not isinstance(hypotheses, dict) or set(hypotheses) != set(TOPICS):
        return "REVIEW_MODEL_OUTPUT"
    supporting = []
    seen = set()
    for topic, assessment in hypotheses.items():
        if not isinstance(assessment, dict) or set(assessment) != {"holat", "dalil"}:
            return "REVIEW_MODEL_OUTPUT"
        status, refs = assessment["holat"], assessment["dalil"]
        if status not in ("mos", "qarshi", "noma_lum") or not isinstance(refs, list):
            return "REVIEW_MODEL_OUTPUT"
        if (status == "noma_lum" and refs) or (status != "noma_lum" and not refs):
            return "REVIEW_MODEL_OUTPUT"
        for ref in refs:
            if not isinstance(ref, str) or ref not in records or ref in seen:
                return "REVIEW_MODEL_OUTPUT"
            seen.add(ref)
        if status == "mos":
            supporting.append(topic)
    if len(supporting) > 1:
        return ACTIONS["ziddiyat"]
    if not supporting:
        return ACTIONS["yetarli_emas"]
    return ACTIONS[supporting[0]]

for row in train+test:
    assert choose_action(row["expected"], row["records"], False) != "REVIEW_MODEL_OUTPUT"
assert choose_action(None, {}, True) == "SHOW_EXISTING_ALERT"
assert choose_action(None, {}, False) == "REVIEW_MODEL_OUTPUT"


In [ ]:
from pathlib import Path
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments,
)

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle Notebook sozlamalarida GPU'ni yoqing.")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
output = Path("/kaggle/working/astromed")
output.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": 0},
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    ),
)

In [ ]:
def evaluate(model):
    model.eval()
    results = []
    with torch.inference_mode():
        for row in test:
            tokens = tokenizer.apply_chat_template(
                row["messages"], add_generation_prompt=True,
                tokenize=True, return_tensors="pt", return_dict=True,
            ).to(model.device)
            sequence = model.generate(
                **tokens, max_new_tokens=260, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )[0, tokens.input_ids.shape[1]:]
            raw = tokenizer.decode(sequence, skip_special_tokens=True).strip()
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                parsed = None
            records = row["records"]
            expected_action = choose_action(row["expected"], records, False)
            action = choose_action(parsed, records, False)
            statuses = parsed.get("ehtimollar", {}) if isinstance(parsed, dict) else {}
            if not isinstance(statuses, dict):
                statuses = {}
            matches = sum(
                statuses.get(key) == value
                for key, value in row["expected"]["ehtimollar"].items()
            )
            results.append({
                "family": row["family"], "expected": row["expected"], "raw": raw,
                "expected_action": expected_action, "action": action,
                "action_match": action == expected_action, "hypotheses_correct": matches,
                "all_correct": matches == len(TOPICS) and action == expected_action,
                "false_specific_action": expected_action in ("ASK_CONTEXT", "REVIEW_CONFLICT")
                and action in ("CHECK_CABIN", "CHECK_INTAKE", "VERIFY_SENSOR", "CHECK_FLIGHT"),
            })
    return results

before = evaluate(model)


In [ ]:
def encode(row):
    prompt = tokenizer.apply_chat_template(
        row["messages"], tokenize=True, add_generation_prompt=True,
    )
    answer = json.dumps(row["expected"], ensure_ascii=False)
    full = tokenizer.apply_chat_template(
        row["messages"] + [{"role": "assistant", "content": answer}], tokenize=True,
    )
    assert full[:len(prompt)] == prompt
    assert len(prompt) < len(full) <= 768
    return {"input_ids": full, "labels": [-100] * len(prompt) + full[len(prompt):]}

def collate(batch):
    width = max(len(row["input_ids"]) for row in batch)
    columns = {"input_ids": [], "attention_mask": [], "labels": []}
    for row in batch:
        count = len(row["input_ids"])
        pad = width - count
        columns["input_ids"].append(row["input_ids"] + [tokenizer.pad_token_id] * pad)
        columns["attention_mask"].append([1] * count + [0] * pad)
        columns["labels"].append(row["labels"] + [-100] * pad)
    return {name: torch.tensor(values, dtype=torch.long) for name, values in columns.items()}

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, target_modules="all-linear",
    bias="none", task_type="CAUSAL_LM",
))
model.config.use_cache = False
model.print_trainable_parameters()

trainer = Trainer(
    model=model, train_dataset=[encode(row) for row in train], data_collator=collate,
    args=TrainingArguments(
        output_dir=str(output / "checkpoints"), per_device_train_batch_size=1,
        gradient_accumulation_steps=2, num_train_epochs=3, learning_rate=1e-4,
        gradient_checkpointing=True, optim="adamw_torch", save_strategy="no",
        report_to="none", logging_steps=5, seed=42,
        fp16=compute_dtype == torch.float16, bf16=compute_dtype == torch.bfloat16,
    ),
)
trainer.train()
model.config.use_cache = True
model.save_pretrained(output / "adapter")
tokenizer.save_pretrained(output / "adapter")

In [ ]:
from shutil import make_archive

after = evaluate(model)
families = {row["family"] for row in after}
for label, rows in (("Bazaviy", before), ("QLoRA", after)):
    print(label, {
        "action": sum(row["action_match"] for row in rows),
        "all_correct": sum(row["all_correct"] for row in rows),
        "false_specific_action": sum(row["false_specific_action"] for row in rows),
        "cases": len(rows),
        "complete_counterfactual_groups": sum(
            all(row["all_correct"] for row in rows if row["family"] == family)
            for family in families if not family.endswith(":missing")
        ),
    }, flush=True)
for filename, data in (
    ("comparison.json", {"model": model_id, "before": before, "after": after}),
    ("synthetic_train.json", train),
    ("synthetic_test.json", test),
):
    (output / filename).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print(make_archive("/kaggle/working/AstroMed_V3_DEMO", "zip", root_dir=output))


Sinov natijalari sintetik ssenariylarga tegishli. Dalil IDsi mavjudligini tekshirish uning mazmuni to‘g‘ri talqin qilinganini kafolatlamaydi. Real missiyada klinik va tizim verifikatsiyasi talab qilinadi.
